In [33]:
import json
from kafka import KafkaProducer

In [34]:
def json_serializer(data):
    return json.dumps(data).encode('utf-8')

In [35]:
server = 'localhost:9092'

In [24]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

producer.bootstrap_connected()

True

In [25]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz

--2026-03-03 15:17:58--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/ea580e9e-555c-4bd0-ae73-43051d8e7c0b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-03T15%3A57%3A53Z&rscd=attachment%3B+filename%3Dgreen_tripdata_2019-10.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-03T14%3A57%3A19Z&ske=2026-03-03T15%3A57%3A53Z&sks=b&skv=2018-11-09&sig=oJKnaOB4tuaIl16x2CRtA3Wxk1R%2FBRBvx1kJJRdapWY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjU1MTM3OCwibmJmIjoxNzcyNTUxMDc4LCJwYXRoIj

In [36]:
import pandas as pd
from time import time

In [37]:
df = pd.read_csv('green_tripdata_2019-10.csv.gz', compression='gzip')

/tmp/ipykernel_69987/2137435753.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('green_tripdata_2019-10.csv.gz', compression='gzip')


In [38]:
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount'
]

In [39]:
df = df[columns]

In [40]:
df['passenger_count'] = df['passenger_count'].fillna(0)

In [41]:
topic_name = 'green-trips'

In [42]:
t0 = time()

for _, row in df.iterrows():
    message = row.to_dict()
    producer.send(topic_name, value=message)

producer.flush()

t1 = time()
took = t1 - t0

print(f"It took {took} sec")

It took 70.0760850906372 sec
